# Pre-Ship Tuning for Deployment Stack

**Date:** 2026-04-18.
**Context:** Validated stack per `findings/stratified_training_investigation.md` §10.8: `combined_score (α=0.5, σ_gap=8) + bandwidth_cap (ceil=0.7d) + piecewise (F=0.7 for ship, per brainstorm G1)`. Before library integration, tighten `σ_gap` and `n_training` parameters per `brainstorm/brainstorm_pre_ship_tuning.md` and `PROMPTS.md` Prompt 4.

**What this notebook does:**

1. Anchor baseline — MAE of the deployment stack at T-3d, T-5d, T-7d (full window to midnight UTC of close, piecewise expansion symmetric).
2. Gap-distribution diagnostic — decides whether tight-σ_gap is feasible.
3. **T1** — σ_gap sweep ∈ {2, 4, 8, 16, ∞} at three snaps. Bootstrap CI (1000 resamples, paired per target). Decision: replace σ_gap=8 if ≥3% T-3d MAE improvement with CI95 lower > 0.
4. **T2** — n_training sweep ∈ {5, 10, 15, 20, 25, 30, 50} using T1's σ_gap winner. Same decision rule.
5. Ship decision — update §1.5 values if a winner emerges, or confirm via CI.

Helpers imported from `_helpers.py` (factored from `stratified_training_validation.ipynb`). Results cached to `.cache/pre_ship_tuning.pkl`.

In [ ]:
import sys
from pathlib import Path

# Make sibling `_helpers` importable when the notebook runs in `notebooks/`
ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, movies, close_date_map, gaps, gap_lookup, first_review_ts,
    gap_for_slug,
    matched_training_slugs, combined_score_selector, gap_overlap_ranked_selector,
    snapshot_state, actual_remaining, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped, predict_window,
    passes_skip_rules_for_snap, bootstrap_mae_delta,
    CACHE_DIR,
)

# Ship-time deployment stack parameters
SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SHIP_F = 0.7
SNAPS = [3.0, 5.0, 7.0]

PRE_SHIP_CACHE_PATH = CACHE_DIR / 'pre_ship_tuning.pkl'

print(f'Cohort: {len(close_date_map)} resolved movies, {len(gaps)} with valid gaps')
print(f'Ship params: combined_score(alpha={SHIP_ALPHA}, sigma_gap={SHIP_SIGMA_GAP}), '
      f'n={SHIP_N_TRAINING}, ceil={SHIP_BANDWIDTH_CEIL}d, F={SHIP_F}')
print(f'Snaps: T-3d, T-5d, T-7d  |  Cache: {PRE_SHIP_CACHE_PATH.name}')

## Core LOO loop

Single function parameterized by (`snap_dbc`, `sigma_gap`, `n_training`). All runs use the ship stack (`alpha=0.5`, `ceil=0.7`, `F=0.7`, snap-adaptive skip rule). Caches by config key.

Prediction per target:
```
phase1 = predict_window(KDE_capped, dbc_from=snap_dbc, dbc_to=0, observed, ...)
phase2 = F * mean(close_day_count(s) for s in training)
predicted_total = phase1 + phase2

actual_total = actual_remaining(target, snap_dbc) + F * close_day_count(target)
```

In [ ]:
def tuning_loo(snap_dbc, sigma_gap, n_training, force=False, verbose=True):
    """LOO across all valid targets for a single (snap_dbc, sigma_gap, n_training) config.

    Stores phase1_pred, actual_remaining, close_day_count(target), mean_close_day_training
    so F can be varied post-hoc if needed. Ship F=0.7 is applied at aggregation time.
    """
    sigma_label = 'inf' if np.isinf(sigma_gap) else f'{sigma_gap:g}'
    config_key = f'snap={snap_dbc:g}_sigma={sigma_label}_n={n_training}'

    cached = pd.read_pickle(PRE_SHIP_CACHE_PATH) if PRE_SHIP_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            if verbose:
                print(f'  [cache] {config_key}: {int((cached["config"] == config_key).sum())} rows')
            return cached[cached['config'] == config_key].copy()

    if verbose:
        print(f'  [run]   {config_key}...')
    rows = []
    skip_log = {'no_obs': 0, 'low_first_review_dbc': 0, 'low_critics': 0}
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, reason = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            if reason and 'no obs' in reason:
                skip_log['no_obs'] += 1
            elif reason and 'first_review_dbc' in reason:
                skip_log['low_first_review_dbc'] += 1
            else:
                skip_log['low_critics'] += 1
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        training, _ = combined_score_selector(
            target, target_gap, target_critics, target_window_days,
            k=n_training, alpha=SHIP_ALPHA, sigma_gap=sigma_gap,
        )
        if len(training) < 5:
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'config': config_key,
                'phase1_pred': np.nan, 'actual_remaining': np.nan,
                'mean_cd_training': np.nan, 'cd_target': np.nan,
            })
            continue

        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )
        phase1 = predict_window(
            model, dbc_from=snap_dbc, dbc_to=0.0,
            observed_critics=target_critics,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        mean_cd = float(np.mean([close_day_count(s) for s in training]))
        cd_t = close_day_count(target)
        rows.append({
            'target_slug': target, 'target_gap': target_gap,
            'snap_dbc': snap_dbc, 'config': config_key,
            'phase1_pred': float(phase1),
            'actual_remaining': int(actual_remaining(target, snap_dbc)),
            'mean_cd_training': mean_cd,
            'cd_target': int(cd_t),
        })

    if verbose:
        kept = len(rows)
        print(f'    kept {kept}, skipped: no_obs={skip_log["no_obs"]} '
              f'low_first_review_dbc={skip_log["low_first_review_dbc"]} '
              f'low_critics={skip_log["low_critics"]}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(
            ['target_slug', 'snap_dbc', 'config'], keep='last',
        )
    else:
        df_combined = df
    df_combined.to_pickle(PRE_SHIP_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()


def apply_F(df, F=SHIP_F):
    """Add F-weighted close-day adjustment to predicted and actual."""
    df = df.dropna(subset=['phase1_pred', 'actual_remaining']).copy()
    df['predicted'] = df['phase1_pred'] + F * df['mean_cd_training']
    df['actual'] = df['actual_remaining'] + F * df['cd_target']
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()
    return df

## Cell 2 — Anchor baseline

Deployment stack at each snap. This is the comparison point for every experiment below.

In [ ]:
anchor_frames = {}
for snap in SNAPS:
    anchor_frames[snap] = tuning_loo(
        snap_dbc=snap, sigma_gap=SHIP_SIGMA_GAP, n_training=SHIP_N_TRAINING,
    )

anchor_summary = []
for snap in SNAPS:
    df = apply_F(anchor_frames[snap], F=SHIP_F)
    anchor_summary.append({
        'snap': f'T-{snap:g}d',
        'n': len(df),
        'MAE': df['abs_err'].mean(),
        'median_err': df['err'].median(),
        'median_abs_err': df['abs_err'].median(),
    })
anchor_df = pd.DataFrame(anchor_summary)
print('=== Anchor baseline: combined_score(alpha=0.5, sigma_gap=8) + ceil=0.7 + piecewise(F=0.7) ===')
print(anchor_df.to_string(index=False, float_format='%.3f'))

## Cell 3 — Gap-distribution diagnostic

For each target, count candidates (past-resolved movies, excluding target) at several `|gap_diff|` thresholds. Stratified by target gap quantile Q1-Q4. Decides whether tight-σ_gap (2-4) can find ≥20 neighbors for most targets.

In [ ]:
gap_quantiles = gaps['gap_days'].quantile([0.25, 0.5, 0.75]).values
q_cutoffs = {'Q1': gap_quantiles[0], 'Q2': gap_quantiles[1], 'Q3': gap_quantiles[2]}

def gap_bucket(g):
    if g <= q_cutoffs['Q1']:
        return 'Q1'
    elif g <= q_cutoffs['Q2']:
        return 'Q2'
    elif g <= q_cutoffs['Q3']:
        return 'Q3'
    return 'Q4'

diag_rows = []
for target in close_date_map:
    target_gap = gap_for_slug(target)
    if target_gap is None:
        continue
    target_close = close_date_map[target]
    cand = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
    ]
    diffs = (cand['gap_days'] - target_gap).abs()
    diag_rows.append({
        'target': target,
        'target_gap': target_gap,
        'bucket': gap_bucket(target_gap),
        'n_candidates_total': len(cand),
        'n_within_0.5d': int((diffs <= 0.5).sum()),
        'n_within_1d': int((diffs <= 1.0).sum()),
        'n_within_2d': int((diffs <= 2.0).sum()),
        'n_within_5d': int((diffs <= 5.0).sum()),
    })

diag = pd.DataFrame(diag_rows)
print(f'Cohort: {len(diag)} targets. Gap quantile cutoffs: '
      f'Q1<={q_cutoffs["Q1"]:.2f}d, Q2<={q_cutoffs["Q2"]:.2f}d, Q3<={q_cutoffs["Q3"]:.2f}d')
print()

# Summary per bucket: median (and pct targets with >=20 candidates) at each band
summary_rows = []
for b in ['Q1', 'Q2', 'Q3', 'Q4']:
    sub = diag[diag['bucket'] == b]
    row = {'bucket': b, 'n_targets': len(sub)}
    for band in ['0.5d', '1d', '2d', '5d']:
        col = f'n_within_{band}'
        row[f'median_{band}'] = int(sub[col].median())
        row[f'pct_ge20_{band}'] = round((sub[col] >= 20).mean() * 100, 1)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
print('Median count of candidates by bucket × band (pct of bucket with >=20 candidates in parens):')
display_cols = ['bucket', 'n_targets']
for band in ['0.5d', '1d', '2d', '5d']:
    summary[band] = summary.apply(
        lambda r: f'{r[f"median_{band}"]:3d} ({r[f"pct_ge20_{band}"]:.0f}%)', axis=1,
    )
    display_cols.append(band)
print(summary[display_cols].to_string(index=False))

print()
print('Cohort-wide fraction with >=20 candidates within:')
for band_col, label in [('n_within_0.5d', '0.5d'), ('n_within_1d', '1d'),
                         ('n_within_2d', '2d'), ('n_within_5d', '5d')]:
    pct = (diag[band_col] >= 20).mean() * 100
    print(f'  |gap_diff| <= {label}: {pct:.1f}%')

## Cell 4 — T1: σ_gap sweep

Sweep σ_gap ∈ {2, 4, 8, 16, ∞} at T-3d, T-5d, T-7d. Holds `n_training=20`, `alpha=0.5`, `ceil=0.7`, `F=0.7`. σ_gap=∞ is pure Jaccard ranking.

**Decision rule:** for any σ_gap to beat 8, need ≥3% T-3d MAE improvement with bootstrap CI95 lower > 0.

In [ ]:
SIGMA_GRID = [2.0, 4.0, 8.0, 16.0, np.inf]

t1_frames = {}
for snap in SNAPS:
    print(f'--- T-{snap:g}d ---')
    for sigma in SIGMA_GRID:
        t1_frames[(snap, sigma)] = tuning_loo(
            snap_dbc=snap, sigma_gap=sigma, n_training=SHIP_N_TRAINING,
        )

In [ ]:
def summarize_sweep(frames, sweep_label, sweep_values, control_value, snaps=SNAPS, F=SHIP_F):
    """Aggregate MAE + bootstrap paired delta vs control. Shared between T1 and T2."""
    lines = [f'=== {sweep_label} sweep ===']
    summary_rows = []

    for snap in snaps:
        control_df = apply_F(frames[(snap, control_value)], F=F)
        control_errs = control_df.set_index('target_slug')['abs_err']

        lines.append('')
        lines.append(f'T-{snap:g}d snap:')
        lines.append(f'  {sweep_label:10s}   n     MAE    delta    pct    CI95_lo  CI95_hi   sig')

        for v in sweep_values:
            v_label = 'inf' if (isinstance(v, float) and np.isinf(v)) else f'{v:g}'
            df = apply_F(frames[(snap, v)], F=F)
            v_errs = df.set_index('target_slug')['abs_err']
            common_idx = control_errs.index.intersection(v_errs.index)
            deltas = (control_errs.loc[common_idx] - v_errs.loc[common_idx]).values
            point, lo, hi = bootstrap_mae_delta(deltas)
            pct = point / control_errs.loc[common_idx].mean() * 100 if len(common_idx) else np.nan
            sig = 'SIG' if (lo > 0 and not np.isnan(lo)) else ('ctrl' if v == control_value else 'ns')
            lines.append(f'  {v_label:10s} {len(df):3d}  {df["abs_err"].mean():6.3f}  '
                        f'{point:+6.3f}  {pct:+5.1f}%  {lo:+7.3f}  {hi:+7.3f}  {sig}')
            summary_rows.append({
                'snap': f'T-{snap:g}d', sweep_label: v_label, 'n': len(df),
                'MAE': df['abs_err'].mean(), 'delta_vs_ctrl': point,
                'pct_vs_ctrl': pct, 'CI95_lo': lo, 'CI95_hi': hi, 'sig': sig,
            })

    print('\n'.join(lines))
    return pd.DataFrame(summary_rows)


t1_summary = summarize_sweep(
    t1_frames, sweep_label='sigma_gap', sweep_values=SIGMA_GRID,
    control_value=SHIP_SIGMA_GAP,
)

In [ ]:
# T1 decision: any σ_gap with ≥3% T-3d MAE improvement AND CI95_lo > 0 wins
print('=== T1 decision rule: ≥3% T-3d MAE improvement AND CI95_lo > 0 ===')
t1_t3 = t1_summary[t1_summary['snap'] == 'T-3d'].copy()
print(t1_t3[['sigma_gap', 'MAE', 'pct_vs_ctrl', 'CI95_lo', 'sig']]
      .to_string(index=False, float_format='%.3f'))

t1_winners = t1_t3[
    (t1_t3['sigma_gap'] != 'inf')
    & (t1_t3['pct_vs_ctrl'] >= 3.0)
    & (t1_t3['CI95_lo'] > 0)
]
t1_winners_inf = t1_t3[
    (t1_t3['sigma_gap'] == 'inf')
    & (t1_t3['pct_vs_ctrl'] >= 3.0)
    & (t1_t3['CI95_lo'] > 0)
]
t1_winners_all = pd.concat([t1_winners, t1_winners_inf])

if len(t1_winners_all) == 0:
    T1_WINNER = SHIP_SIGMA_GAP
    print(f'\nNo σ_gap beats 8 at the ≥3% + CI95_lo>0 bar. Keep σ_gap=8 for T2.')
else:
    # Pick the biggest improvement
    best = t1_winners_all.loc[t1_winners_all['pct_vs_ctrl'].idxmax()]
    winner_label = best['sigma_gap']
    T1_WINNER = float('inf') if winner_label == 'inf' else float(winner_label)
    print(f'\nT1 WINNER: sigma_gap={winner_label} (+{best["pct_vs_ctrl"]:.1f}%, CI95_lo={best["CI95_lo"]:+.3f})')

print(f'\nUsing sigma_gap={"inf" if np.isinf(T1_WINNER) else f"{T1_WINNER:g}"} for T2.')

## Cell 5 — T2: n_training sweep

Using T1's σ_gap winner, sweep `n_training ∈ {5, 10, 15, 20, 25, 30, 50}` at T-3d, T-5d, T-7d. Same decision rule.

In [ ]:
N_GRID = [5, 10, 15, 20, 25, 30, 50]

t2_frames = {}
for snap in SNAPS:
    print(f'--- T-{snap:g}d ---')
    for n in N_GRID:
        t2_frames[(snap, n)] = tuning_loo(
            snap_dbc=snap, sigma_gap=T1_WINNER, n_training=n,
        )

In [ ]:
t2_summary = summarize_sweep(
    t2_frames, sweep_label='n_training', sweep_values=N_GRID,
    control_value=SHIP_N_TRAINING,
)

In [ ]:
# T2 decision rule: ≥3% T-3d MAE improvement AND CI95_lo > 0
print('=== T2 decision rule: ≥3% T-3d MAE improvement AND CI95_lo > 0 ===')
t2_t3 = t2_summary[t2_summary['snap'] == 'T-3d'].copy()
print(t2_t3[['n_training', 'MAE', 'pct_vs_ctrl', 'CI95_lo', 'sig']]
      .to_string(index=False, float_format='%.3f'))

t2_winners = t2_t3[
    (t2_t3['n_training'] != f'{SHIP_N_TRAINING}')
    & (t2_t3['pct_vs_ctrl'] >= 3.0)
    & (t2_t3['CI95_lo'] > 0)
]

if len(t2_winners) == 0:
    T2_WINNER = SHIP_N_TRAINING
    print(f'\nNo n_training beats {SHIP_N_TRAINING} at the ≥3% + CI95_lo>0 bar. Keep n=20.')
else:
    best = t2_winners.loc[t2_winners['pct_vs_ctrl'].idxmax()]
    T2_WINNER = int(best['n_training'])
    print(f'\nT2 WINNER: n_training={T2_WINNER} (+{best["pct_vs_ctrl"]:.1f}%, CI95_lo={best["CI95_lo"]:+.3f})')

## Cell 6 — Ship decision

Final recommendation: which parameter values go into `BACKLOG.md` §1.5.

In [ ]:
final_sigma_label = 'inf' if np.isinf(T1_WINNER) else f'{T1_WINNER:g}'
print('=== SHIP DECISION ===')
print(f'  sigma_gap:   {final_sigma_label}    (was 8)  {"CHANGED" if T1_WINNER != SHIP_SIGMA_GAP else "unchanged"}')
print(f'  n_training:  {T2_WINNER}     (was 20) {"CHANGED" if T2_WINNER != SHIP_N_TRAINING else "unchanged"}')
print(f'  alpha:       0.5   (unchanged — α sweep at noise floor per findings §10.7)')
print(f'  ceil:        0.7d  (unchanged)')
print(f'  F:           0.7   (unchanged — G1 re-estimation pending fresh data)')
print(f'  floor:       0.5d  (unchanged)')
print()
print('Update BACKLOG.md §1.5 only if any of sigma_gap or n_training changed.')